# 📘 Geospatial Code Refactoring - First Iteration

---

## 📝 1. Introduction

The purpose of this notebook is to clearly and simply demonstrate how to refactor a Python script according to specific best practices. These practices will help you write code that is modular, understandable, and easy to reuse.

### 🌍 Example from Geospatial Analysis

We will work with a practical geospatial example:

**Calculate land-use statistics within administrative boundaries.**

#### 📌 **Given:**
- A polygon shapefile representing administrative boundaries (such as districts or cities).
- A raster dataset showing various land-use categories.

#### 🎯 **Objective:**
- Determine the percentage area covered by each land-use category within each administrative boundary.

Throughout this notebook, we’ll first show the algorithm in its original (non-refactored) form, clearly highlighting its limitations. Then, we'll progressively refactor the code following specific guidelines.

---



In [ ]:
# =========================================
# 2. Original (Non-refactored) Code
# =========================================

import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np

# Global variables (hardcoded paths)
shapefile_path = "data/admin_boundaries.shp"
raster_path = "data/land_use.tif"

# Function using global variables (incorrect usage)
def load_data(shapefile, raster_path):
    gdf = gpd.read_file(shapefile_path)
    raster = rasterio.open(raster_path)
    return gdf, raster

# Function with hardcoded local values (incorrect usage)
def calculate_percentages(masked_data, non_data_value):
    nodata_value = -9999  # Hardcoded value
    unique, counts = np.unique(masked_data[masked_data != nodata_value], return_counts=True)
    total_pixels = np.sum(counts)
    return {int(cls): (count / total_pixels) * 100 for cls, count in zip(unique, counts)}

# Main logic directly in script (incorrect structure)
gdf, raster = load_data()
results = []
for index, row in gdf.iterrows():
    geometry = [row['geometry']]
    masked_raster, transform = rasterio.mask.mask(raster, geometry, crop=True)
    masked_data = masked_raster[0]
    percentages = calculate_percentages(masked_data)
    results.append({
        'admin_area': row['NAME'],
        'land_use_percentages': percentages
    })

# Results directly printed in script (without main guard)
for result in results:
    print(result)


# 🎯 Refactoring Goals for This Iteration

The objectives we aim to achieve through refactoring are:

1. **Modular Functions**: Convert script logic into clear, reusable, and modular functions.
2. **Eliminate Global Variables**: Replace global variables by passing them as function parameters.
3. **Remove Hardcoded Values**: Define previously hardcoded values as function parameters with sensible default settings.
4. **Encapsulation of Logic**: Wrap the main algorithm logic within a callable function.
5. **Safe Execution**: Ensure script execution only when intended using the main guard `if __name__ == "__main__":`.
6. **Functionality Testing**: Verify that the script continues to work correctly after refactoring.


In [ ]:
# =========================================
# Refactored Code According to Rules
# =========================================

import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np

# Corrected: Function to load shapefile now takes path as parameter (removed global variable)
def load_shapefile(shapefile_path):
    return gpd.read_file(shapefile_path)

# Corrected: Function to load raster now takes path as parameter (removed global variable)
def load_raster(raster_path):
    return rasterio.open(raster_path)

# Corrected: Function to calculate percentages now uses parameter for nodata value (no hardcoded value)
def calculate_percentages(masked_data, nodata_value=-9999):
    valid_data = masked_data[masked_data != nodata_value]
    unique, counts = np.unique(valid_data, return_counts=True)
    total_pixels = np.sum(counts)
    return {int(cls): (count / total_pixels) * 100 for cls, count in zip(unique, counts)}

# Corrected: Entire main logic encapsulated into a single callable function (improves modularity and readability)
def perform_land_use_analysis(shapefile_path, raster_path, nodata_value=-9999):
    gdf = load_shapefile(shapefile_path)
    raster = load_raster(raster_path)

    results = []
    for _, row in gdf.iterrows():
        geometry = [row['geometry']]
        masked_raster, _ = rasterio.mask.mask(raster, geometry, crop=True)
        masked_data = masked_raster[0]
        percentages = calculate_percentages(masked_data, nodata_value)
        results.append({
            'admin_area': row['NAME'],
            'land_use_percentages': percentages
        })

    return results

# Corrected: Used main guard to prevent unintended execution upon import
if __name__ == "__main__":
    shapefile_path = "data/admin_boundaries.shp"  # defined at the entry point for clarity
    raster_path = "data/land_use.tif"  # defined at the entry point for clarity

    analysis_results = perform_land_use_analysis(shapefile_path, raster_path)

    # Clearly separated result printing from the core analysis logic
    for result in analysis_results:
        print(result)

#### If you want to execute your code in Jyputer Notebook, you can call directly your functions without using them in the main

In [ ]:
shapefile_path = "data/admin_boundaries.shp"
raster_path = "data/land_use.tif"

# Explicitly call your main function
analysis_results = perform_land_use_analysis(shapefile_path, raster_path)

# Print results
for result in analysis_results:
    print(result)

## 📌 Rules of Thumb for Creating and Organizing Functions

To help you refactor effectively and write clearer, reusable code, follow these **Rules of Thumb**:

- 🔄 **Repeated Code Rule:** If code appears more than once, move it into a function.
- 🎯 **Single Responsibility Rule:** Each function should perform exactly one clear task.
- 📖 **Readability Rule:** If your function is difficult to understand quickly (e.g., longer than ~30 lines), break it down into smaller, clearly named functions.
- 🧩 **Reuse and Testing Rule:** If code will likely be reused or needs testing independently, encapsulate it in a separate function.
- 📌 **Naming Rule:** Clearly name functions based on their specific task, such as `load_shapefile`, `calculate_land_use_percentages`, or `mask_raster_to_polygon`.

Applying these rules consistently will help your team maintain a high standard of code quality and readability.


## 🚩 How to Identify if a Function is Too Big
⚠️   **Length Symptom:**
The function exceeds roughly 30 lines, making it hard to understand its purpose quickly.
Action: Break the function into smaller functions, each handling a specific subtask.

⚠️  **Multiple Responsibilities Symptom**: The function handles several unrelated tasks (e.g., reading files, processing, calculations, saving).
Action: Divide the function into separate functions based on each clear responsibility.

⚠️  **Multiple Levels of Nested Loops or Conditions Symptom** : Deeply nested loops or conditional statements make the logic overly complex.
Action: Extract inner logic into dedicated functions to simplify control flow.

⚠️  **Lack of Error Handling Symptom**: The function does not handle exceptions or edge cases, increasing the risk of bugs.
Action: Isolate error handling into smaller blocks or functions to keep the main logic clean.



# General Tips 
-📌 **Keep Functions Focused:**  
  Each function should have one clear purpose. If you can’t describe it in one sentence, consider breaking it down further.

-📌 **Consistent Naming Conventions:**  
  Use names like `load_data`, `calculate_percentages`, and `mask_raster` to clearly reflect the function’s role.

-📌 **Minimize Side Effects:**  
  Functions should avoid modifying global state or relying on external variables without explicitly passing them as parameters.

-📌 **Error Handling:**  
  Integrate error checks and handle exceptions within functions to prevent unexpected crashes.

-📌 **Avoid Deep Nesting:**  
  Refactor code with excessive nested loops or conditionals to improve clarity and maintainability.



# General Tips for Python Functions

- 📌 **Keep Functions Focused:**  
  Each function should have one clear purpose. If you can’t describe it in one sentence, consider breaking it down further.

- 📌 **Consistent Naming Conventions:**  
  Follow a clear and descriptive naming standard to improve readability and maintainability:  
  - Use **lowercase_with_underscores** for function names (e.g., `load_data`, `calculate_percentages`, `mask_raster`).  
  - Use verbs that describe the action the function performs (`get_value`, `process_image`, `compute_statistics`).  
  - For boolean-returning functions, use `is_` or `has_` (e.g., `is_valid_input`, `has_permission`).  
  - Keep names concise but meaningful—avoid generic names like `do_something`.  

- 📌 **Minimize Side Effects:**  
  Functions should avoid modifying global state or relying on external variables without explicitly passing them as parameters.

- 📌 **Error Handling:**  
  Integrate error checks and handle exceptions within functions to prevent unexpected crashes. Use `try/except` blocks where needed and raise meaningful errors.

- 📌 **Avoid Deep Nesting:**  
  Refactor code with excessive nested loops or conditionals to improve clarity and maintainability. Break down complex logic into smaller helper functions when necessary.